# Chapter 2

Add your content here.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MoERouter(nn.Module):
    def __init__(self, input_dim, num_experts, top_k=2):
        super().__init__()
        self.gate_weight = nn.Linear(input_dim, num_experts)
        self.top_k = top_k
        
    def forward(self, x):
        # x shape: [batch, input_dim]
        
        # 1. Calculate Router Logits
        logits = self.gate_weight(x)
        
        # 2. Calculate Probabilities (Softmax)
        probs = F.softmax(logits, dim=-1)
        
        # 3. Select Top-K Experts
        # values: the probability scores
        # indices: the ID of the expert (0, 1, 2...)
        top_k_probs, top_k_indices = torch.topk(probs, self.top_k, dim=-1)
        
        # 4. Re-normalize probabilities
        # The chosen experts' weights should sum to 1 for stability
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)
        
        return top_k_indices, top_k_probs

# Example Usage
router = MoERouter(input_dim=128, num_experts=8, top_k=2)
dummy_input = torch.randn(1, 128) # One token
indices, weights = router(dummy_input)

print(f"Selected Expert IDs: {indices.tolist()}")
print(f"Routing Weights: {weights.tolist()}")

Selected Expert IDs: [[2, 4]]
Routing Weights: [[0.6005693674087524, 0.39943063259124756]]
